# WaterTAP tour — facet edition

A re-implementation of the classic `watertap-1` client walkthrough using the new
**facet-based** query API (`aq.graph()`), on the seawater-RO `model.ttl` in this folder.

Two symmetric moves plus introspection:

| move | meaning | rows |
|------|---------|------|
| `.facets()` | show what predicates/objects are reachable next | (read-only) |
| `.refine(step, **φ)` | **stay** on the current nodes, keeping those with such an edge | never multiplies |
| `.pivot(step, **φ)` | **move** the cursor to the neighbours along an edge | adds a column |

A **`Profile`** curates the discovery surface (hide noise, name virtual edges). Results come out
with `.count()` / `.nodes()` / `.frame()` / `.select(...)`; inspect with `.to_sparql()`.

> **Scope:** graframe is the *metadata / graph* plane. Timeseries pull + unit conversion (the last
> third of `watertap-1`) still live on the classic `acq.find_*` / `DataObject` API; the final
> section shows the bridge.

## Connect

In [ ]:
import polars as pl
from acquirium import Acquirium
from acquirium.Graframe import P, Profile, Reasoning

pl.Config.set_fmt_str_lengths(70)
pl.Config.set_tbl_rows(30)

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)
# g (the Graframe root) is built in the profile section below.

## Load the model

`insert_graph` reads the file client-side (path relative to this notebook).

In [ ]:
acq.insert_graph("model.ttl", format="turtle", replace=True)
print("graph version:", acq.graph_version())

## Curate the view with a profile

An ontology exposes far more predicates than any one task cares about. A `Profile` shapes the
*discovery surface* — which predicates/types show up in facets — and lets you **name virtual
edges** (property paths) so you traverse `pivot("downstream")` instead of
`pivot(P(connectedTo).plus())`.

`Profile.base()` hides schema noise (`rdf`/`rdfs`/`owl`/`sh`, class/shape objects); we layer the
water-domain predicates and a few named paths on top. Profiles shape discovery only — you can
still `pivot`/`refine` a hidden predicate explicitly, or pass `raw=True` to a facet call.

In [ ]:
water = Profile.base().with_(
    # predicates worth seeing (namespace globs + a few exacts):
    allow=["s223:", "nawi:", "qudt:hasQuantityKind", "qudt:hasUnit", "s223:ofSubstance"],
    # ...minus the low-level connection plumbing:
    deny=[
        "s223:cnx", "s223:connected", "s223:connectedThrough", "s223:hasConnectionPoint",
        "s223:isConnectionPointOf", "s223:hasBoundaryConnectionPoint", "s223:connectsAt",
        "s223:connectsThrough", "s223:connectsTo", "s223:connectsFrom", "s223:connectedFrom",
    ],
    # named virtual edges (paths this domain actually cares about):
    edges={
        "downstream": "s223:connectedTo+",                      # transitive flow
        "upstream":   "^s223:connectedTo+",
        "measures":   "s223:hasProperty",                      # equipment to property
        "quantity":   "s223:hasProperty/qudt:hasQuantityKind", # equipment to quantity kind
    },
)

g = acq.graph(profile=water)

## Find entities by class

`g.instances(cls)` is the seed; it includes subclasses by default (the reasoning profile).

In [ ]:
pumps = g.instances("nawi:Pump")
print("pumps:", pumps.count())
pumps.frame()

In [ ]:
# Raw firehose vs. the profiled view (named virtual edges surface at the top):
pumps.facets(raw=True).show(12)   # everything the ontology exposes
pumps.facets().show()             # curated + named edges (downstream/measures/...)

## Follow relationships

`pivot` walks an edge. Because the profile named `downstream = s223:connectedTo+`, you traverse
it by name — no `P(...)` in sight. Filter the far end inline with `is_a=` / `value=`.

In [ ]:
# Everything reachable downstream of pump P1, and just the static mixers among them:
print("reachable downstream of P1:", g.nodes("wbs:P1").pivot("downstream").count())
g.nodes("wbs:P1").pivot("downstream", is_a="nawi:StaticMixer").frame()

## Data nodes (observable properties)

The measurable "data" are `s223:QuantifiableObservableProperty` nodes, attached to equipment via
`s223:hasProperty` (the named `measures` edge) and `observe`d by sensors.

In [ ]:
props = g.instances("s223:QuantifiableObservableProperty")
print("observable properties:", props.count())
props.facets(by="predicate", direction="out").show()

In [ ]:
# The observable properties attached to a pump, via the named "measures" edge:
g.nodes("wbs:P1").pivot("measures").frame()

## Filter data nodes

`refine` narrows the current set by an edge condition. The classic `filter_by_quantity_kind` /
`filter_by_unit` / `filter_by_substance` become refinements on the property's edges.

In [ ]:
# See what's actually available to filter on:
props.facets(by="pred-obj", direction="out", limit=40).to_polars().filter(
    pl.col("predicate").is_in(["qudt:hasQuantityKind", "qudt:hasUnit", "s223:ofSubstance"])
)

In [ ]:
print("Pressure           :", props.refine("qudt:hasQuantityKind", value="qk:Pressure").count())
print("unit kg/s          :", props.refine("qudt:hasUnit", value="unit:KiloGM-PER-SEC").count())

salt_flow = (props
    .refine("s223:ofSubstance", value="nawi:Constituent-Salt")
    .refine("qudt:hasUnit", value="unit:KiloGM-PER-SEC"))
print("salt mass flow (kg/s):", salt_flow.count())
salt_flow.frame()

## Inspect the query

Every selection compiles to SPARQL — no black box.

In [ ]:
print(salt_flow.to_sparql())

## Systems

Systems are logical groupings of equipment/junctions/subsystems (`s223:hasMember`).

In [ ]:
g.instances("s223:System").frame()

In [ ]:
# Hierarchy: systems that are members of other systems
(g.instances("s223:System").mark("system")
   .pivot("s223:hasMember", is_a="s223:System").mark("subsystem")
   .select("system", "subsystem"))

In [ ]:
# Equipment count per system (direct members that are Equipment)
by_system = (g.instances("s223:System").mark("system")
              .pivot("s223:hasMember", is_a="s223:Equipment").mark("equipment"))
(by_system.select("system", "equipment")
          .group_by("system")
          .agg(pl.col("equipment").count().alias("equipment_count"))
          .sort("equipment_count", descending=True))

In [ ]:
# Pumps that are members of a specific system
g.nodes("wbs:pretreatment-system").pivot("s223:hasMember", is_a="nawi:Pump").frame()

## All pumps and their observed properties

A join built with waypoints: mark the pump, hop out via `measures` to its properties (and their
quantity kind), mark those, then `select` the columns you want.

In [ ]:
(g.instances("nawi:Pump").mark("pump")
   .pivot("measures").mark("property")
   .pivot("qudt:hasQuantityKind").mark("quantity")
   .select("pump", "property", "quantity"))

## All data-generating entities within a system

System members → their properties → each property's quantity kind. (Desalination system, whose
equipment carry the observable properties directly.)

In [ ]:
(g.nodes("wbs:desalination-system")
   .pivot("s223:hasMember", is_a="s223:Equipment").mark("equipment")
   .pivot("measures").mark("property")
   .pivot("qudt:hasQuantityKind").mark("quantity")
   .select("equipment", "property", "quantity"))

## Bridging to timeseries

Graframe answers *which points*; hand `.nodes()` to the classic data API for the *values*
(timeseries + unit conversion), e.g. `acq.find_entity(uri=...).find_data().dataframe(...)`.
(A native `.data()` on selections is the planned next step.)

In [ ]:
pressure_points = props.refine("qudt:hasQuantityKind", value="qk:Pressure").nodes()
pressure_points